# Stage 2 Notebook 77 - Joint BDD training with CULane-pretrained KD teacher (from NB74)

**The real KD experiment.** NB72 self-distilled from NB62 (same architecture, similar quality) and got nothing useful -- garbage teacher in. NB77 uses a TRUE pretrained teacher: the lane head produced by NB74 (trained on CULane, the standard lane benchmark, with no BDD overlap).

Two reasons this should help where NB72 didn't:
1. The teacher saw a DIFFERENT data distribution (CULane = Chinese highways; BDD = US urban). Distillation transfers the teacher's curve-fitting prior, which has been refined on data the student has never seen.
2. CULane pretrain has only lane supervision (no det conflict). The teacher's lane head is in a better local minimum than any joint-trained checkpoint we have.

Note: NB76 separately downloads CLRKDNet's published GitHub checkpoint (DLA-34, 80.87 CULane F1) as a stronger alternative teacher. NB77 uses NB74's checkpoint because the head architecture matches our student exactly (CLRKDLaneHead -> CLRKDLaneHead). Using CLRKDNet's published weights would require an architecture adapter (DLA-34 backbone vs our RMT-GCA, slightly different head shape) -- doable as Exp2TTT but more engineering. NB77 is the immediate path.

Speed flags applied (NB73 proved them stable):
- `batch_size: 32`, `lr0: 4e-4` (sqrt-scaled from baseline 8 / 2e-4)
- `workers: 6`, `prefetch_factor: 4`, `torch.compile(reduce-overhead)`
- `backbone_lr_mult: 0.01` (joint-conflict throttle)
- `cls_separate_path: true` (NB62 baseline)

### Run mode
1. Cell 2 mounts Drive and installs deps.
2. Cell 3 extracts NB74's CULane-pretrain tar and locates the teacher checkpoint.
3. Cell 4 trains the BDD joint model with KD enabled.

Prerequisite: NB74 finished successfully and produced `/content/drive/MyDrive/EcoCAR/training_runs/exp69_rmt_gca_culane_lane_only_pretrain_culane8.tar`. If NB74 failed (as the first run did due to the CULane archive path issue), re-run NB74 first.

In [ ]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [ ]:
# Step 1: extract NB74's tar and find the teacher checkpoint.
from pathlib import Path
import os, sys, tarfile, shutil

NB74_TAR = '/content/drive/MyDrive/EcoCAR/training_runs/exp69_rmt_gca_culane_lane_only_pretrain_culane8.tar'
EXTRACT_DIR = '/content/exp69_rmt_gca_culane_lane_only_pretrain_culane8'

if not Path(NB74_TAR).exists():
    raise FileNotFoundError(
        f'NB74 output {NB74_TAR} does not exist. Run NB74 first '
        '(make sure cell 3 finds the CULane archives -- see NB74 cell 3 diagnostics).'
    )

if not Path(EXTRACT_DIR).exists() or not any(Path(EXTRACT_DIR).iterdir()):
    print('Extracting', NB74_TAR, '->', EXTRACT_DIR)
    Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)
    with tarfile.open(NB74_TAR, 'r') as tar:
        tar.extractall(EXTRACT_DIR)
else:
    print('Already extracted')

# Find the teacher .pt file -- look for best.pt or last.pt in the extracted tree.
candidates = list(Path(EXTRACT_DIR).rglob('best.pt')) + list(Path(EXTRACT_DIR).rglob('last.pt'))
if not candidates:
    raise FileNotFoundError(
        f'No best.pt or last.pt under {EXTRACT_DIR}. NB74 may have crashed before saving.'
    )
TEACHER_PT = str(candidates[0])
print('Teacher checkpoint:', TEACHER_PT)
print('Size:', Path(TEACHER_PT).stat().st_size / 1e6, 'MB')

# Show what's inside the checkpoint -- helps debug architecture mismatch later.
import torch
ck = torch.load(TEACHER_PT, map_location='cpu', weights_only=False)
state = ck.get('state_dict', ck.get('model', ck))
if hasattr(state, 'state_dict'):
    state = state.state_dict()
if isinstance(state, dict):
    lane_keys = [k for k in state.keys() if 'lane_head' in k][:5]
    print(f'top-level keys: {len(state)}; lane_head sample: {lane_keys}')

Extracting /content/drive/MyDrive/EcoCAR/training_runs/exp69_rmt_gca_culane_lane_only_pretrain_culane8.tar -> /content/exp69_rmt_gca_culane_lane_only_pretrain_culane8


/tmp/ipykernel_1751/827930566.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_DIR)


Teacher checkpoint: /content/exp69_rmt_gca_culane_lane_only_pretrain_culane8/best.pt
Size: 95.669385 MB
top-level keys: 827; lane_head sample: ['_orig_mod.lane_head.prior_ys', '_orig_mod.lane_head.sample_ys', '_orig_mod.lane_head.lateral.0.0.weight', '_orig_mod.lane_head.lateral.0.1.weight', '_orig_mod.lane_head.lateral.0.1.bias']


In [3]:
# Step 2: train BDD joint model with NB74 teacher distillation enabled.
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp70_rmt_gca_anchor_cls_sep_vfl_culane_kd_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 32
    LIMIT_TRAIN = None
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

# Patch the config so teacher.lane_head_checkpoint points at the path we
# discovered in cell 3. We write a patched yaml to /content so the
# original config in the repo stays untouched.
import yaml
with open(CONFIG, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg.setdefault('teacher', {})['lane_head_checkpoint'] = TEACHER_PT
PATCHED_CONFIG = f'/content/{Path(CONFIG).stem}_patched.yaml'
with open(PATCHED_CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f)
print('Patched config wrote teacher.lane_head_checkpoint =', TEACHER_PT)

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', PATCHED_CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
    '--workers', '6',
    '--prefetch-factor', '4',
    '--torch-compile',
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('Teacher:', TEACHER_PT, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

Patched config wrote teacher.lane_head_checkpoint = /content/exp69_rmt_gca_culane_lane_only_pretrain_culane8/best.pt
DEBUG_MODE: False
Teacher: /content/exp69_rmt_gca_culane_lane_only_pretrain_culane8/best.pt
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config /content/exp70_rmt_gca_anchor_cls_sep_vfl_culane_kd_full_data_joint_patched.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp70_rmt_gca_anchor_cls_sep_vfl_culane_kd_full_data_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp70_rmt_gca_anchor_cls_sep_vfl_culane_kd_full_data_joint_full12.tar --epochs 12 --batch-size 32 --limit-val 1000 --force-extract --print-every 50 --workers 6 --prefetch-factor 4 --torch-compile
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp70_rmt_gca_anchor_cls_sep_vfl_culane_kd_full_data_joint_full12.tar
Visible log file: /content/drive/MyDri

0

## What to watch in NB77

Reference NB73 (no KD, same speed flags, 12 ep full): matched_iou=0.558, oracle_f1=0.484, decoded_f1=0.050, val_lane_best_f1=0.110, gap=0.029.
Reference NB72 (NB62-self-distill): matched_iou=0.549, decoded_f1=0.048 -- KD with bad teacher = no change.

Pass criteria at epoch 12:
- **`val/lane/distill` decreases monotonically** -- teacher signal is active and useful.
- `val/matched_line_iou >= 0.56` -- preserve or beat NB73's geometry.
- **`val/lane_best_f1 >= 0.15`** -- 1.5x NB73; CULane-pretrained teacher's cls knowledge transfers.
- **`val/lane/decoded_f1 >= 0.07`** -- 1.5x NB73; the smoking gun for KD actually working.
- `pos_score - neg_score >= 0.04` -- KD helps cls separation past the anchor-head ceiling.

Failure signals:
- distill loss flat or rising: teacher checkpoint corrupted or arch mismatch silently failing.
Verify cell 3's print of lane_head keys.
- decoded_f1 unchanged from NB73: teacher trained on CULane doesn't transfer to BDD.
Confirms NB72-style 'self-distill is useless' conclusion -- KD needs domain overlap.
Next move would be NB76's CLRKDNet weights with a proper architecture adapter (Exp2TTT).